In [6]:
!pip install pyspark==3.5.1

Note: you may need to restart the kernel to use updated packages.


In [10]:
import os
import sys
import re

# =========================================================
# 1. ÉP VERSION PYSPARK (PHẢI CHẠY ĐẦU TIÊN TRƯỚC KHI IMPORT)
# =========================================================
modules_to_remove = [mod for mod in sys.modules if mod.startswith('pyspark') or mod.startswith('py4j')]
for mod in modules_to_remove: 
    del sys.modules[mod]

sys.path = [p for p in sys.path if "/usr/local/spark" not in p]
if "PYTHONPATH" in os.environ: 
    del os.environ["PYTHONPATH"]
    
conda_site_packages = "/opt/conda/lib/python3.13/site-packages"
if conda_site_packages not in sys.path: 
    sys.path.insert(0, conda_site_packages)
    
os.environ["SPARK_HOME"] = os.path.join(conda_site_packages, "pyspark")
os.environ["PYSPARK_PYTHON"] = "python3"
os.environ["PYSPARK_DRIVER_PYTHON"] = "python3"


# =========================================================
# 2. BÂY GIỜ MỚI IMPORT PYSPARK VÀ KHỞI TẠO SESSION
# =========================================================
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, to_date

spark = SparkSession.builder \
    .appName("DataCleaning_Pipeline") \
    .config("spark.jars.packages", "org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.262,org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.5.0") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "dataNLPmining-lab") \
    .config("spark.hadoop.fs.s3a.secret.key", "dataNLPmining-lab") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider") \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config("spark.sql.catalog.my_catalog", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.my_catalog.type", "hadoop") \
    .config("spark.sql.catalog.my_catalog.warehouse", "s3a://raw-financial-data/iceberg_warehouse") \
    .getOrCreate()


# =========================================================
# 3. VÁ LỖI THỜI GIAN HADOOP
# =========================================================
hadoop_conf = spark._jsc.hadoopConfiguration()
iterator = hadoop_conf.iterator()
while iterator.hasNext():
    entry = iterator.next()
    val = str(entry.getValue()).strip().lower()
    match = re.fullmatch(r"(\d+)([smhd])", val)
    if match:
        num, unit = int(match.group(1)), match.group(2)
        ms_val = num * 1000 if unit == 's' else num * 60000 if unit == 'm' else num * 3600000 if unit == 'h' else num * 86400000
        hadoop_conf.set(entry.getKey(), str(ms_val))

print("✅ Khởi tạo Spark và môi trường hoàn tất!")

✅ Khởi tạo Spark và môi trường hoàn tất!


In [11]:
spark.sql("CREATE NAMESPACE IF NOT EXISTS my_catalog.raw_zone")

# --- PIPELINE NEWS ---
path_news = "s3a://raw-financial-data/raw_zone/news_data/"
df_news_clean = spark.read.format("json") \
    .option("multiLine", "true").option("recursiveFileLookup", "true") \
    .option("mode", "DROPMALFORMED").option("inferSchema", "true").load(path_news) \
    .select(col("id"), col("content.title").alias("title"), col("content.summary").alias("summary"), col("content.pubDate").alias("published_at"))

df_news_clean.write.format("iceberg").mode("overwrite").saveAsTable("my_catalog.raw_zone.news_data_clean")

# --- PIPELINE MARKET ---
path_market = "s3a://raw-financial-data/raw_zone/market_data/"
df_market_clean = spark.read.format("json") \
    .option("multiLine", "true").option("recursiveFileLookup", "true") \
    .option("mode", "DROPMALFORMED").option("inferSchema", "true").load(path_market) \
    .select(
        to_date(col("Date")).alias("trade_date"), col("Open").alias("open_price"), col("High").alias("high_price"),
        col("Low").alias("low_price"), col("Close").alias("close_price"), col("Volume").alias("volume"),
        col("Dividends").alias("dividends"), col("`Stock Splits`").alias("stock_splits"), col("`Capital Gains`").alias("capital_gains")
    )

df_market_clean.write.format("iceberg").mode("overwrite").saveAsTable("my_catalog.raw_zone.market_data_clean")
print("🎉 Đã lưu xong dữ liệu vào Raw Zone!")

🎉 Đã lưu xong dữ liệu vào Raw Zone!
